<a href="https://colab.research.google.com/github/OmerAkhan01/moltook-agent-behavior-analysis/blob/veri-hazirlama/Moltbook_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import ast

# Moltbook ham veri setinin Parquet formatında sisteme yüklenmesi.
dosya_adi = 'train-00000-of-00001.parquet'

try:
    df = pd.read_parquet(dosya_adi)
    print(f"Veri başarıyla okundu. Toplam satır sayısı: {len(df)}")
except Exception as e:
    print(f"Dosya okuma hatası! Lütfen dosyanın dizinde mevcut olduğunu kontrol edin. Hata: {e}")

Veri başarıyla okundu. Toplam satır sayısı: 44376


In [ ]:
# 'post' sütunundaki yarı-yapılandırılmış (nested) verileri ayrıştıran fonksiyon.
def post_parcala(x):
    try:
        # String formatındaki veriyi güvenli bir şekilde Python objesine (dictionary) dönüştürür.
        data = ast.literal_eval(str(x))

        # Analiz için gerekli 'content' ve 'upvotes' özniteliklerinin çıkarılması.
        return pd.Series({
            'content_body': data.get('content', 'Veri Yok'),
            'upvotes_count': data.get('upvotes', 0)
        })
    except:
        # Veri kirliliği veya format bozukluğu durumunda hata yakalama süreci.
        return pd.Series({'content_body': 'Hata: Format Bozuk', 'upvotes_count': 0})

print("Veri düzleştirme (Flattening) işlemi başlatıldı...")
post_detaylari = df['post'].apply(post_parcala)

Veri düzleştirme (Flattening) işlemi başlatıldı...


In [ ]:
# Veri bütünlüğünü sağlamak amacıyla yazar kimliklerinin (author_id) eşleştirilmesi.
if 'author' in df.columns:
    df['author_id'] = df['author']
elif 'user_id' in df.columns:
    df['author_id'] = df['user_id']
else:
    df['author_id'] = "Tanımlanamadı"

# İşlenmiş verilerin orijinal etiketler ile birleştirilerek final tablosunun oluşturulması.
df_final = pd.concat([df[['id', 'topic_label', 'toxic_level', 'author_id']], post_detaylari], axis=1)

# Nihai veri setinin dışa aktarılması.
df_final.to_csv('moltbook_final_v4.csv', index=False)

print("-" * 30)
print("Süreç Tamamlandı: moltbook_final_v4.csv dosyası oluşturuldu.")

------------------------------
Süreç Tamamlandı: moltbook_final_v4.csv dosyası oluşturuldu.
